# Small Plummer Sphere on GPU

This notebook samples a Plummer sphere, runs an adaptive Hermite solve,
and reports simple conservation diagnostics on the active JAX device.

In [ ]:
import time

import jax
import jax.numpy as jnp

from nornax import (
    AarsethController,
    initialize_state,
    sample_plummer_sphere,
    solve_adaptive_to_time,
    total_angular_momentum,
    total_energy,
)
from nornax.forces import DirectSumGravity

jax.config.update("jax_enable_x64", True)
jax.devices()

In [ ]:
n = 256
order = 8
t_final = 0.1
positions, velocities, masses = sample_plummer_sphere(jax.random.PRNGKey(0), n)
force_model = DirectSumGravity()
controller = AarsethController(eta=0.05, min_dt=1.0e-5, max_dt=1.0e-2)

reference = initialize_state(positions, velocities, masses, force_model, max_order=4)
e0 = float(total_energy(reference))
l0 = total_angular_momentum(reference)

In [ ]:
@jax.jit
def run():
    return solve_adaptive_to_time(
        positions,
        velocities,
        masses,
        force_model,
        t_final=t_final,
        order=order,
        controller=controller,
        atol=1.0e-6,
    )

warmup = run()
warmup.final_state.positions.block_until_ready()
t0 = time.perf_counter()
result = run()
result.final_state.positions.block_until_ready()
t1 = time.perf_counter()

ef = float(total_energy(result.final_state))
lf = total_angular_momentum(result.final_state)

{
    "accepted_steps": int(result.dt_history.shape[0]),
    "elapsed_seconds": t1 - t0,
    "energy_drift": abs(ef - e0),
    "angular_momentum_drift": float(jnp.linalg.norm(lf - l0)),
}